# CS781 Phase 5 (statute identification) — PART 2 of 2: restore Part 1, experiments E4 and E5, ensemble, rule check, final prediction.json

**Run Part 1 first.** Part 2 needs the dataset `cs781-phase5-lsi-results` that Part 1 uploaded: attach it (Add Input > Your datasets) **before** pressing Save & Run All, together with the Phase 1 datasets (BiLSTM and GPT-2 are needed here; E5 also needs the E1 and E6 checkpoints, which come from the backup). Same Secrets and settings as Part 1 (Internet ON, GPU, `HF_TOKEN`, `KAGGLE_USERNAME`, `KAGGLE_KEY`).

| cells | content |
|---|---|
| config, setup, credentials, toolkit, data | the same as Part 1 (the data is loaded again; no baselines or window test: their results come back with the ledger and the saved window order) |
| tokenisation, models, adapters, trainer | the same as Part 1 |
| **restore** | copies Part 1's checkpoints, results, submissions, ledger and window order back, and restores `ORDER` |
| E4 | recurrent (BiLSTM), partial fine-tuning (512-token windows, 2 epochs, so it fits the session) |
| E5 | knowledge distillation: teachers E1 and E6 are scored on the 42.8k training cases, then the 6-layer GPT-2 student is trained (and the no-distillation control) |
| extras | a LoRA fallback on GPT-2 only if no LoRA experiment exists (for example if T5 never learned) |
| salvage, **rule check**, backup, output | the best single run or ensemble written to `prediction.json`; the check of the capstone rules over all six experiments from both parts; `requirements_report.md`; upload; output check |

Estimated time on a T4 (rough): preamble 0.3 h, restore 0.1 h, E4 about 2.3 h, E5 about 3 h, salvage + rule check + backup 0.5 h: about 6-7 hours.
The rule check counts distinct (model, technique) pairs over the restored ledger plus this part's runs: E1, E6, E2, E3 from Part 1 and E4, E5 here. If it reports 'NOT YET COMPLIANT', its message names what is missing.


In [ ]:
# ================= CELL 0 — CONFIG  [RUNS NOW]. Edit this cell only; nothing below needs editing. =================
HF_DATASET = "Exploration-Lab/CS-781-Capstone"                       # gated Hugging Face dataset repo (accept its conditions on the page first)
HF_CONFIG = "phase_5"                                                # the subset / config name of the Phase 5 data
HF_SPLITS = {"train": "train", "dev": "validation", "test": "test"}  # our name -> the split name in the dataset (dev = the official validation split)
DATA_SOURCE = "hf"                                                   # "hf" = load_dataset(HF_DATASET, HF_CONFIG); "files" = read train / validation / test json(l) files from LOCAL_DATA_DIR
LOCAL_DATA_DIR = None                                                # e.g. "/kaggle/input/my-phase5-files"; also the fallback if the Hugging Face load fails
ID_KEY, TEXT_KEY, LABEL_KEY, PRED_KEY = "id", "text", "labels", "pred"   # field names in the data; PRED_KEY is the key of the submission objects: [{"id": "...", "pred": [int, ...]}]
CKPT_PATHS = {"gpt2_new": None, "modernbert": None, "t5": None, "bilstm": None}   # path to each Phase 1 final_model.pt; None = find it automatically under /kaggle/input
TOKENIZER_PATH = None                                                # path to legal_bpe_32k_mask.json (inside your Phase 1 dataset); None = find it automatically
LSI_WORK = "/kaggle/working/lsi"                                     # results, ledger, submissions
CKPT_DIR = "/kaggle/working/lsi/checkpoints"                         # where the FINE-TUNED checkpoints are saved
SAVE_MODELS = True                                                   # save a checkpoint for every experiment run
MACRO_CONVENTION = "present_in_val"                                  # which macro-F1 drives threshold tuning and model selection: "present_in_val" or "universe" (both are always reported)
TEXT_KEEP_CHARS = 60000                                              # characters kept from the start and from the end of every document (enough for 4,096 tokens at each end)


In [ ]:
# ================= CELL 1 — setup (Phase 5: Legal Statute Identification) =================
import os, sys, re, json, math, time, random, glob, gc, copy, hashlib, pickle, getpass, zipfile, shutil
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

SEED = 42
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); AMP = device.type == "cuda"
WORK = Path(LSI_WORK); (WORK / "runs").mkdir(parents=True, exist_ok=True); (WORK / "cache").mkdir(exist_ok=True); SUBDIR = WORK / "submissions"; SUBDIR.mkdir(exist_ok=True)
LEDGER = WORK / "ledger.csv"
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working"]
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu (training cells need a GPU)", "| work dir:", WORK)


In [ ]:
# ===== CREDENTIALS CHECK  [RUNS NOW; it never asks you to type anything]: reads HF_TOKEN (and optionally KAGGLE_USERNAME / KAGGLE_KEY) from Kaggle Secrets, verifies them, and stops early with the exact fix =====
# Needs Internet ON. Add the Secrets under Add-ons > Secrets with 'Attached' ON for THIS notebook (a committed run only sees attached secrets). The Hugging Face token is not needed when DATA_SOURCE = "files".
import os, subprocess, shutil
REQUIRE_HF, REQUIRE_KAGGLE = (DATA_SOURCE == "hf"), False        # the Kaggle key is only needed for the optional backup upload (set REQUIRE_KAGGLE = True to insist)
def get_secret(name):
    v = os.environ.get(name)
    if not v:
        try:
            from kaggle_secrets import UserSecretsClient; v = UserSecretsClient().get_secret(name)
        except Exception: v = None
    return (v or "").strip().strip("'\"") or None
problems, report = [], []
hf = get_secret("HF_TOKEN"); HF_OK = False
if DATA_SOURCE != "hf": report.append("Hugging Face token : not needed (DATA_SOURCE = 'files')"); HF_OK = True
elif not hf: report.append("Hugging Face token : MISSING"); problems.append("HF_TOKEN is missing: add a Secret named HF_TOKEN (a token of the account that was approved for the dataset), or set DATA_SOURCE = 'files'")
else:
    try:
        from huggingface_hub import HfApi; api = HfApi(); me = api.whoami(token=hf); api.auth_check(HF_DATASET, repo_type="dataset", token=hf); HF_OK = True; os.environ["HF_TOKEN"] = hf
        report.append(f"Hugging Face token : OK (user '{me.get('name')}', access to {HF_DATASET} confirmed, {len(hf)} characters)")
    except Exception as e:
        code = getattr(getattr(e, "response", None), "status_code", None)
        why = "Hugging Face does not recognise it (wrong, expired or revoked: create a new classic 'Read' token and update the Secret)" if code == 401 else "valid token, but this account has no approved access to the gated dataset" if code == 403 else f"{type(e).__name__}: {str(e)[:100]}"
        report.append(f"Hugging Face token : REFUSED (HTTP {code}): {why}"); problems.append(f"HF_TOKEN does not work: {why}")
ku, kk = get_secret("KAGGLE_USERNAME"), get_secret("KAGGLE_KEY"); KAGGLE_OK = False
if not (ku and kk): report.append("Kaggle credentials  : not provided (the backup upload will be a dry run)"); (problems.append("KAGGLE_USERNAME / KAGGLE_KEY are missing") if REQUIRE_KAGGLE else None)
elif not shutil.which("kaggle"): report.append("Kaggle credentials  : cannot be verified (the kaggle command is not installed)"); (problems.append("the kaggle command is not installed") if REQUIRE_KAGGLE else None)
else:
    try:
        r = subprocess.run(["kaggle", "datasets", "list", "--mine"], capture_output=True, text=True, timeout=120, env={**os.environ, "KAGGLE_USERNAME": ku, "KAGGLE_KEY": kk})
        if r.returncode == 0: KAGGLE_OK = True; os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = ku, kk; report.append(f"Kaggle credentials  : OK (user '{ku}')")
        else: report.append(f"Kaggle credentials  : REFUSED ({(r.stdout + r.stderr).strip()[:100]})"); (problems.append("the Kaggle username/key were refused (use the LEGACY API key; Internet ON?)") if REQUIRE_KAGGLE else None)
    except Exception as e: report.append(f"Kaggle credentials  : not verified ({type(e).__name__})")
print("\n".join(report))
blocking = [p for p in problems if (p.startswith("HF_TOKEN") and REQUIRE_HF) or (not p.startswith("HF_TOKEN") and REQUIRE_KAGGLE)]
if blocking: raise RuntimeError("STOPPING EARLY (before any long step):\n  - " + "\n  - ".join(blocking) + "\nFix: Add-ons > Secrets > add the missing secret(s), switch 'Attached' ON for this notebook, and run again.")
print("\nCredentials ready.")


In [ ]:
# ================= CELL 3 — EVALUATION TOOLKIT  [RUNS NOW; numpy only]. Every metric is defined here, in the open. The official evaluator's convention is NOT known, so nothing here claims to be it. =================
# Y and P are 0/1 matrices [cases, statutes] over the label universe. Per statute: F1 = 2TP / (2TP + FP + FN).
#   macro_f1_present   : mean F1 over the statutes that occur in Y (support > 0). A statute that is predicted but never occurs counts as a false positive only through micro-F1, not here.
#   macro_f1_universe  : mean F1 over ALL statutes of the label universe; a statute with TP + FP + FN = 0 (never occurs, never predicted) gets F1 = 0   (scikit-learn: zero_division=0)
#   macro_f1_universe_zd1 : the same, but such a statute gets F1 = 1  (scikit-learn: zero_division=1). It is a sensitivity bound, shown so that you can see how much the convention matters.
#   micro_f1           : F1 from the summed TP / FP / FN.   subset_acc : the share of cases whose predicted set equals the true set exactly.
CONVENTIONS = {"present_in_val": "macro_f1_present", "universe": "macro_f1_universe"}
assert MACRO_CONVENTION in CONVENTIONS, f"MACRO_CONVENTION must be one of {list(CONVENTIONS)}"
SEL_KEY = CONVENTIONS[MACRO_CONVENTION]
GRID = np.round(np.arange(0.05, 0.951, 0.01), 2); MIN_SUP, SHRINK, PL_MARGIN = 5, 0.5, 0.003

def sigmoid(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))
def evaluate(Y, P):
    Y = Y.astype(bool); P = P.astype(bool); tp = (Y & P).sum(0).astype(float); fp = (~Y & P).sum(0).astype(float); fn = (Y & ~P).sum(0).astype(float); den = 2 * tp + fp + fn
    f0 = np.divide(2 * tp, den, out=np.zeros_like(den), where=den > 0); f1 = np.where(den > 0, f0, 1.0); sup = Y.sum(0); TP, FP, FN = tp.sum(), fp.sum(), fn.sum()
    return dict(macro_f1_present=float(f0[sup > 0].mean()) if (sup > 0).any() else 0.0, macro_f1_universe=float(f0.mean()), macro_f1_universe_zd1=float(f1.mean()), micro_f1=float(2 * TP / max(1.0, 2 * TP + FP + FN)), subset_acc=float((Y == P).all(1).mean()),
                micro_p=float(TP / max(1.0, TP + FP)), micro_r=float(TP / max(1.0, TP + FN)), n_present=int((sup > 0).sum()), n_universe=int(Y.shape[1]), f1_per_label=f0, support=sup)
def f1_scores(Y, P): m = evaluate(Y, P); m["macro_f1"] = m[SEL_KEY]; return m                 # 'macro_f1' = the convention chosen in MACRO_CONVENTION (used for tuning and selection)
def average_precision(y, s):
    if y.sum() == 0: return float("nan")
    o = np.argsort(-s, kind="stable"); y = y[o].astype(float); tp = np.cumsum(y); return float((tp / np.arange(1, len(y) + 1))[y > 0].mean())
def macro_ap(Y, S):
    aps = [average_precision(Y[:, j], S[:, j]) for j in range(Y.shape[1]) if Y[:, j].sum() > 0]; return float(np.mean(aps)) if aps else 0.0
def with_top1(P, S):
    P = P.copy(); empty = ~P.any(1)
    if empty.any(): P[empty, S[empty].argmax(1)] = True
    return P
def tune_global(S, Y):
    best_f, best_t = -1.0, 0.5
    for t in GRID:
        f = f1_scores(Y, with_top1(S >= t, S))["macro_f1"]
        if f > best_f + 1e-12: best_f, best_t = f, float(t)
    return best_t, best_f
def tune_per_label(S, Y, tau_g):
    Yb = Y.astype(bool); sup = Yb.sum(0); f = np.zeros((len(GRID), S.shape[1]))
    for gi, t in enumerate(GRID):
        P = S >= t; tp = (Yb & P).sum(0).astype(float); den = 2 * tp + (~Yb & P).sum(0) + (Yb & ~P).sum(0); f[gi] = np.divide(2 * tp, den, out=np.zeros_like(den, dtype=float), where=den > 0)
    best = GRID[f.argmax(0)]; tau = np.full(S.shape[1], tau_g); ok = sup >= MIN_SUP; tau[ok] = SHRINK * best[ok] + (1 - SHRINK) * tau_g; return tau
def card_threshold(S, target): return float(np.quantile(S.ravel(), 1.0 - min(0.999, target / S.shape[1])))
def select_policy(S, Y, card_target, seed=0):
    """THRESHOLDS ARE TUNED ON VALIDATION DATA ONLY. S = validation probabilities. Policies: a global threshold; a cardinality-matched threshold (mean statutes per case equals the validation mean; it uses scores only, never test labels);
       per-statute thresholds shrunk toward the global one (scored with an honest 2-fold estimate on validation, chosen only if it wins by PL_MARGIN). Every case always gets at least its top-scoring statute."""
    tau_g, _ = tune_global(S, Y); sc = {"global_top1": f1_scores(Y, with_top1(S >= tau_g, S))["macro_f1"]}
    tau_c = card_threshold(S, card_target); sc["card_matched"] = f1_scores(Y, with_top1(S >= tau_c, S))["macro_f1"]
    perm = np.random.default_rng(seed).permutation(len(Y)); folds = np.array_split(perm, 2); Pcv = np.zeros(Y.shape, dtype=bool)
    for k in (0, 1):
        a, b = folds[1 - k], folds[k]; tg, _ = tune_global(S[a], Y[a]); tl = tune_per_label(S[a], Y[a], tg); Pcv[b] = with_top1(S[b] >= tl[None, :], S[b])
    sc["per_label"] = f1_scores(Y, Pcv)["macro_f1"]; tau_l = tune_per_label(S, Y, tau_g); best = max(("global_top1", "card_matched"), key=lambda k: sc[k])
    if sc["per_label"] >= sc[best] + PL_MARGIN: best = "per_label"
    return dict(policy=best, tau=tau_g, tau_l=tau_l, tau_c=tau_c, scores=sc)
def apply_policy(sel, S, card_target):
    p = sel["policy"]
    if p == "global_top1": return with_top1(S >= sel["tau"], S)
    if p == "per_label": return with_top1(S >= sel["tau_l"][None, :], S)
    return with_top1(S >= card_threshold(S, card_target), S)
def paired_bootstrap(Y, Pa, Pb, n=300, seed=0):
    """paired bootstrap over cases of (convention macro-F1 of A) - (that of B): mean difference, 95% interval, and the share of resamples in which A beats B"""
    r = np.random.default_rng(seed); d = []
    for _ in range(n):
        i = r.integers(0, len(Y), len(Y)); d.append(f1_scores(Y[i], Pa[i])["macro_f1"] - f1_scores(Y[i], Pb[i])["macro_f1"])
    d = np.array(d); return float(d.mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)), float((d > 0).mean())

# ---- the submission file: a JSON list [{"id": "<id as in the test file>", "pred": [integer statute ids]}] ----
def to_py(x): return x.item() if hasattr(x, "item") else x
def lists_from(P, logits): return [[int(LABELS[j]) for j in sorted(np.where(P[i])[0], key=lambda j: -logits[i, j])] for i in range(len(P))]
def write_submission_ml(path, rids, label_lists): json.dump([{"id": to_py(r), PRED_KEY: [int(x) for x in ls]} for r, ls in zip(rids, label_lists)], open(path, "w"))
def validate_submission_ml(path, rids):
    sub = json.load(open(path)); assert isinstance(sub, list) and len(sub) == len(rids), f"{path}: expected {len(rids)} predictions, found {len(sub)}"
    for s, r in zip(sub, rids):
        assert set(s) == {"id", PRED_KEY}, f"bad keys {set(s)}"; assert s["id"] == to_py(r), f"id mismatch {s['id']!r} vs {to_py(r)!r}"; ls = s[PRED_KEY]
        assert isinstance(ls, list) and all(type(x) is int for x in ls), "pred must be a list of integers"; assert len(set(ls)) == len(ls), "duplicate statute ids in one prediction"; assert all(x in L2I for x in ls), f"statute id outside the label universe in {ls[:5]}"
    assert len({str(s["id"]) for s in sub}) == len(sub), "duplicate ids"; return True
def mean_ci(vals): return float(np.mean(vals)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))
print(f"macro-F1 convention used for tuning and selection: {MACRO_CONVENTION} ({SEL_KEY}). All three variants are reported everywhere. The official evaluator's convention is unresolved.")


In [ ]:
# ================= CELL 4 — LOAD THE OFFICIAL PHASE 5 DATA AND INSPECT IT  [RUNS NOW: CPU only, no checkpoints] =================
def col(ds, key): return list(ds[key]) if hasattr(ds, "column_names") else [r.get(key) for r in ds]
def has(ds, key): return key in ds.column_names if hasattr(ds, "column_names") else any(key in r for r in list(ds)[:50])
def rows_hf():
    from datasets import load_dataset, get_dataset_config_names
    tk = os.environ.get("HF_TOKEN")
    try: cfgs = get_dataset_config_names(HF_DATASET, token=tk)
    except Exception: cfgs = None
    if cfgs is not None: assert HF_CONFIG in cfgs, f"config '{HF_CONFIG}' not found in {HF_DATASET}; available configs: {cfgs}"
    ds = load_dataset(HF_DATASET, HF_CONFIG, token=tk)
    for v in HF_SPLITS.values(): assert v in ds, f"split '{v}' not in {list(ds.keys())}: fix HF_SPLITS in the config cell"
    return {k: ds[v] for k, v in HF_SPLITS.items()}
def rows_files():
    out = {}
    for sp, names in (("train", ("train",)), ("dev", ("validation", "valid", "val", "dev")), ("test", ("test",))):
        c = [p for p in glob.glob(f"{LOCAL_DATA_DIR}/**/*.json*", recursive=True) if any(n in os.path.basename(p).lower() for n in names)]
        assert c, f"no {sp} file found under {LOCAL_DATA_DIR}"; p = max(c, key=os.path.getsize)
        try: obj = json.load(open(p, encoding="utf-8"))
        except json.JSONDecodeError: obj = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
        if isinstance(obj, dict): obj = next((v for v in obj.values() if isinstance(v, list)), obj)
        out[sp] = obj; print(f"[{sp}] {p}")
    return out
def load_splits():
    if DATA_SOURCE == "hf":
        try: return rows_hf()
        except Exception as e:
            print(f"Hugging Face load failed: {type(e).__name__}: {str(e)[:300]}")
            if not LOCAL_DATA_DIR: raise
            print("-> falling back to LOCAL_DATA_DIR")
    assert LOCAL_DATA_DIR, "DATA_SOURCE = 'files' needs LOCAL_DATA_DIR"; return rows_files()
RAW = load_splits(); print({k: len(v) for k, v in RAW.items()}, "cases per split")

def extract(ds, name):
    ids = [to_py(x) for x in col(ds, ID_KEY)]; texts = col(ds, TEXT_KEY); n_chars = np.array([len(t) for t in texts]); n_words = np.array([t.count(" ") + 1 for t in texts])
    texts = [t if len(t) <= 2 * TEXT_KEEP_CHARS else t[:TEXT_KEEP_CHARS] + " " + t[-TEXT_KEEP_CHARS:] for t in texts]            # keep both ends of very long documents
    labs = None
    if name != "test":
        assert has(ds, LABEL_KEY), f"the {name} split has no '{LABEL_KEY}' field: fix LABEL_KEY in the config cell"
        labs = []
        for i, l in enumerate(col(ds, LABEL_KEY)):
            try: labs.append(sorted({int(x) for x in (l or [])}))
            except (TypeError, ValueError): raise ValueError(f"{name} case {i}: '{LABEL_KEY}' = {l!r} is not a list of integer statute ids (the specification says pred is a list of integer ids)")
    elif has(ds, LABEL_KEY): print("[test] NOTE: the test split has a label field. Test labels are private, so it is IGNORED and never used.")
    return dict(ids=ids, texts=texts, labs=labs, n_chars=n_chars, n_words=n_words)
DATA = {sp: extract(ds, sp) for sp, ds in RAW.items()}; del RAW; gc.collect()
assert len({str(i) for i in DATA["test"]["ids"]}) == len(DATA["test"]["ids"]), "duplicate test ids"
DATA["test"]["labs"] = [[] for _ in DATA["test"]["ids"]]                                                                       # test labels are never used for anything
# ---- the label universe: every statute id seen in train or validation ----
LABELS = sorted({l for sp in ("train", "dev") for ls in DATA[sp]["labs"] for l in ls}); L2I = {l: i for i, l in enumerate(LABELS)}; V = len(LABELS)
def label_matrix(labs):
    Y = np.zeros((len(labs), V), dtype=np.uint8)
    for i, ls in enumerate(labs):
        for l in ls: Y[i, L2I[l]] = 1
    return Y
YTR, YDV = label_matrix(DATA["train"]["labs"]), label_matrix(DATA["dev"]["labs"]); card_dev = float(YDV.sum(1).mean()); card_tr = float(YTR.sum(1).mean())
sup_tr, sup_dv = YTR.sum(0), YDV.sum(0); only_dev = [LABELS[j] for j in range(V) if sup_tr[j] == 0 and sup_dv[j] > 0]; only_train = [LABELS[j] for j in range(V) if sup_dv[j] == 0 and sup_tr[j] > 0]

print(f"\n=== LABEL UNIVERSE: {V} statutes (ids {LABELS[0]}..{LABELS[-1]}) = train ∪ validation | {int((sup_tr > 0).sum())} occur in train, {int((sup_dv > 0).sum())} in validation ===")
print(f"statutes seen ONLY in validation (cannot be learned from train): {len(only_dev)} {only_dev[:10]} | statutes seen only in train: {len(only_train)}")
print(f"labels per case: train mean {card_tr:.2f} / median {np.median(YTR.sum(1)):.0f} / p90 {np.percentile(YTR.sum(1), 90):.0f} / max {YTR.sum(1).max()} | validation mean {card_dev:.2f} | cases with no label: train {np.mean(YTR.sum(1) == 0):.2%}, validation {np.mean(YDV.sum(1) == 0):.2%}")
print(f"train support per statute: min {int(sup_tr.min())} / median {int(np.median(sup_tr))} / max {int(sup_tr.max())} | statutes with < 5 training cases: {int((sup_tr < 5).sum())} | with < 20: {int((sup_tr < 20).sum())}")
order_ = np.argsort(-sup_tr); tab = pd.DataFrame({"statute": [LABELS[j] for j in order_], "train": sup_tr[order_], "validation": sup_dv[order_]}); print("most / least frequent statutes in train:"); print(pd.concat([tab.head(8), tab.tail(5)]).to_string(index=False))
co = (YTR.astype(np.int32).T @ YTR.astype(np.int32)); np.fill_diagonal(co, 0); iu = np.triu_indices(V, 1); top = np.argsort(-co[iu])[:5]; print("most frequent statute pairs in one case (statute, statute, cases):", [(LABELS[iu[0][t]], LABELS[iu[1][t]], int(co[iu][t])) for t in top])
print("\n=== DOCUMENT LENGTH (words, split on spaces) ===")
rows = [dict(split=sp, cases=len(d["ids"]), p50=int(np.percentile(d["n_words"], 50)), p90=int(np.percentile(d["n_words"], 90)), p99=int(np.percentile(d["n_words"], 99)), max=int(d["n_words"].max()), **{f">{b}": f"{np.mean(d['n_words'] > b):.0%}" for b in (512, 2048, 4096)}) for sp, d in DATA.items()]; print(pd.DataFrame(rows).to_string(index=False))
lw = pd.Series(DATA["train"]["n_words"]); nl = pd.Series(YTR.sum(1)); print(f"Spearman correlation between document length and the number of statutes of a case (train): {lw.corr(nl, method='spearman'):.3f}")
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.6)); ax[0].bar(range(V), np.sort(sup_tr)[::-1]); ax[0].set_yscale("log"); ax[0].set_title("train support per statute (sorted)"); ax[1].hist(YTR.sum(1), bins=range(0, int(YTR.sum(1).max()) + 2)); ax[1].set_title("statutes per case (train)")
    ax[2].hist(np.log10(DATA["train"]["n_words"] + 1), bins=50); ax[2].set_title("log10 words per document (train)"); plt.tight_layout(); plt.savefig(WORK / "data_overview.png", dpi=110); plt.close(); print("saved", WORK / "data_overview.png")
except Exception as e: print("plot skipped:", type(e).__name__)


In [ ]:
# ================= CELL 5 — TOKENISE THE DOCUMENTS  [NEEDS the Phase 1 tokenizer file legal_bpe_32k_mask.json; CPU only] =================
# Every case keeps its first 4,096 and last 4,096 tokens (a case of up to 8,192 tokens is kept whole); the neural windows are cut from these.
from tokenizers import Tokenizer
HEAD, TAIL = 4096, 4096
tpaths = [TOKENIZER_PATH] if TOKENIZER_PATH else (sorted(glob.glob("/kaggle/input/**/legal_bpe_32k_mask.json", recursive=True)) or sorted(glob.glob("/kaggle/input/**/legal_bpe_32k*.json", recursive=True)) or sorted(glob.glob("/kaggle/working/**/legal_bpe_32k*.json", recursive=True)))
assert tpaths and os.path.exists(tpaths[0]), "tokenizer not found: attach your Phase 1 dataset (it holds legal_bpe_32k_mask.json) or set TOKENIZER_PATH in the config cell"
tok = Tokenizer.from_file(tpaths[0]); EOS, BOS, PAD = tok.token_to_id("[EOS]"), tok.token_to_id("[BOS]"), tok.token_to_id("[PAD]"); print("tokenizer:", tpaths[0], "| vocabulary", tok.get_vocab_size())
def tokenise(split, bs=128):
    d = DATA[split]; cache = WORK / "cache" / f"toks_{split}_{len(d['ids'])}_{TEXT_KEEP_CHARS}_{tok.get_vocab_size()}.pkl"
    if cache.exists(): d["toks"], d["lens"] = pickle.load(open(cache, "rb")); print(f"[{split}] loaded cache: {len(d['toks'])} cases"); return
    toks, lens = [], []; t0 = time.time()
    for i in range(0, len(d["texts"]), bs):
        for e in tok.encode_batch(d["texts"][i:i + bs], add_special_tokens=False):
            ids = e.ids; n = len(ids); toks.append(np.asarray(ids if n <= HEAD + TAIL else ids[:HEAD] + ids[-TAIL:], dtype=np.int32)); lens.append(n)
    d["toks"], d["lens"] = toks, np.asarray(lens); pickle.dump((d["toks"], d["lens"]), open(cache, "wb"), protocol=4); print(f"[{split}] tokenised {len(toks)} cases in {time.time()-t0:.0f}s")
for sp in DATA: tokenise(sp)
print("\nDOCUMENT LENGTH IN TOKENS (of the kept text; very long documents were trimmed to both ends before tokenising):")
print(pd.DataFrame([dict(split=sp, p50=int(np.percentile(d["lens"], 50)), p90=int(np.percentile(d["lens"], 90)), p99=int(np.percentile(d["lens"], 99)), **{f">{b}": f"{np.mean(d['lens'] > b):.0%}" for b in (512, 2048, 4096, 8192)}) for sp, d in DATA.items()]).to_string(index=False))


In [ ]:
# ================= CELL 6 — LOAD THE PHASE 1 MODELS  [NEEDS the Phase 1 checkpoints; set CKPT_PATHS in the config cell or attach the datasets]. The old 3,000-step GPT-2 is never used. =================
ARCH_LOADERS_SRC = r'''"""arch_loaders.py - load any of the four pre-trained architectures as a backbone for classification / tagging.
   load_backbone(path) -> (backbone, hidden_size, causal, max_len, family)
   backbone(input_ids, attention_mask) -> object with .last_hidden_state of shape [B, T, hidden_size]"""
import torch, torch.nn as nn
from types import SimpleNamespace
from transformers import GPT2Config, GPT2LMHeadModel, ModernBertConfig, ModernBertForMaskedLM, T5Config, T5ForConditionalGeneration

class BiLSTMLM(nn.Module):
    def __init__(self, vocab, emb=512, hidden=1024, layers=3):
        super().__init__()
        self.vocab, self.emb_dim, self.hidden, self.layers = vocab, emb, hidden, layers
        self.emb = nn.Embedding(vocab, emb); self.fwd = nn.LSTM(emb, hidden, layers, batch_first=True); self.bwd = nn.LSTM(emb, hidden, layers, batch_first=True)
        self.norm_f, self.norm_b = nn.LayerNorm(hidden), nn.LayerNorm(hidden)
        self.proj_f, self.proj_b = nn.Linear(hidden, emb), nn.Linear(hidden, emb); self.out_bias = nn.Parameter(torch.zeros(vocab))
    def hidden_states(self, input_ids, attention_mask=None):
        x = input_ids; B, T = x.shape
        lengths = attention_mask.sum(1) if attention_mask is not None else torch.full((B,), T, device=x.device)
        rev = (lengths[:, None] - 1 - torch.arange(T, device=x.device)[None, :]).clamp(min=0)
        hf, _ = self.fwd(self.emb(x)); hb, _ = self.bwd(self.emb(x.gather(1, rev)))
        hb = hb.gather(1, rev[:, :, None].expand(-1, -1, hb.shape[-1]))
        return torch.cat([hf, hb], dim=-1)

class Backbone(nn.Module):
    """one calling convention for every family (matches what the Phase 3 SeqClassifier expects)"""
    def __init__(self, family, inner, embed):
        super().__init__(); self.family, self.inner, self._embed = family, inner, embed
    def get_input_embeddings(self): return self._embed
    def forward(self, input_ids=None, attention_mask=None, inputs_embeds=None, use_cache=False, **kw):
        f = self.family
        if f == "gpt2": h = self.inner(input_ids=input_ids, attention_mask=attention_mask, inputs_embeds=inputs_embeds, use_cache=False).last_hidden_state
        else:
            assert inputs_embeds is None, f"prompt tuning (inputs_embeds) is only wired for the gpt2 family, not {f}"
            if f in ("modernbert", "t5"): h = self.inner(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            elif f == "bilstm": h = self.inner.hidden_states(input_ids, attention_mask)
            else: raise ValueError(f)
        return SimpleNamespace(last_hidden_state=h)

CAUSAL = {"gpt2": True, "modernbert": False, "t5": False, "bilstm": False}
# modules to adapt with LoRA / LoRMA / IA3 (None = full fine-tuning only). ModernBERT's q,k,v are ONE fused linear, 'Wqkv' (rows: q | k | v).
ADAPTER_TARGETS = {"gpt2": "attn.c_attn (fused Conv1D: q|k|v)", "modernbert": "attn.Wqkv (fused Linear: q|k|v)", "t5": "SelfAttention q, k, v, o (Linear)", "bilstm": None}

def build_full(mc):
    f, V, d = mc["family"], mc["vocab_size"], mc["hidden_size"]
    if f == "gpt2":
        return GPT2LMHeadModel(GPT2Config(vocab_size=V, n_positions=mc["context_length"], n_ctx=mc["context_length"], n_embd=d, n_layer=mc["num_layers"], n_head=mc["num_heads"], n_inner=mc.get("n_inner", 4 * d),
                                          resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0))
    if f == "modernbert":
        return ModernBertForMaskedLM(ModernBertConfig(vocab_size=V, max_position_embeddings=mc["context_length"], hidden_size=d, intermediate_size=mc["intermediate_size"], num_hidden_layers=mc["num_layers"],
                                                      num_attention_heads=mc["num_heads"], local_attention=mc["local_attention"], global_attn_every_n_layers=mc["global_attn_every_n_layers"], pad_token_id=0,
                                                      attention_dropout=0.0, mlp_dropout=0.0, embedding_dropout=0.0, classifier_pooling="mean", sparse_prediction=True, sparse_pred_ignore_index=-100, reference_compile=False))
    if f == "t5":
        return T5ForConditionalGeneration(T5Config(vocab_size=V, d_model=d, d_kv=mc["d_kv"], d_ff=mc["d_ff"], num_layers=mc["num_layers"], num_decoder_layers=mc["num_decoder_layers"], num_heads=mc["num_heads"],
                                                   feed_forward_proj="gated-gelu", dropout_rate=0.0, tie_word_embeddings=True))
    if f == "bilstm": return BiLSTMLM(V, emb=mc["emb"], hidden=mc["lstm_hidden"], layers=mc["num_layers"])
    raise ValueError(f)

def load_backbone(path, device="cpu"):
    ck = torch.load(path, map_location="cpu", weights_only=False); mc = ck["model_config"]; full = build_full(mc); full.load_state_dict(ck["model_state_dict"], strict=True)
    f = mc["family"]
    if f == "gpt2": inner, embed = full.transformer, full.transformer.wte
    elif f == "modernbert": inner, embed = full.model, full.model.embeddings.tok_embeddings
    elif f == "t5": inner, embed = full.encoder, full.shared
    elif f == "bilstm": inner, embed = full, full.emb
    else: raise ValueError(f)
    return Backbone(f, inner, embed).to(device), mc["hidden_size"], CAUSAL[f], mc["context_length"], f
'''
al_paths = sorted(glob.glob("/kaggle/input/**/arch_loaders.py", recursive=True)) or sorted(glob.glob("/kaggle/working/**/arch_loaders.py", recursive=True))
if not al_paths:                                                                  # self-contained: the loader module is embedded in this notebook
    (WORK / "arch_loaders.py").write_text(ARCH_LOADERS_SRC); al_paths = [str(WORK / "arch_loaders.py")]; print("arch_loaders.py not found among the inputs: using the copy embedded in this notebook")
sys.path.insert(0, str(Path(al_paths[0]).parent)); import arch_loaders as AL

MODEL_PATHS = {}                                    # filled automatically below; you can also set entries by hand, e.g. MODEL_PATHS["bilstm"] = "/kaggle/input/.../final_model.pt"
def discover_models():
    found = {}
    for p in sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True)):
        if os.path.getsize(p) < 50e6: continue
        base = os.path.basename(p)
        if base != "final_model.pt": continue                # only the NEW Phase 1 models: the old 3,000-step checkpoint is deliberately never used
        try: ck = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
        except Exception: continue
        if not isinstance(ck, dict) or "model_config" not in ck or "model_state_dict" not in ck: continue
        mc = ck["model_config"]; fam = mc.get("family")
        if fam is None: continue                              # a checkpoint without a family entry is an old-format model: skipped
        key = "gpt2_new" if fam == "gpt2" else fam
        if key not in found: found[key] = p
    return found
MODEL_PATHS.update({k: v for k, v in discover_models().items() if k not in MODEL_PATHS}); MODEL_PATHS.update({k: v for k, v in CKPT_PATHS.items() if v})        # CKPT_PATHS from the config cell wins
print("Phase 1 models found:", {k: v for k, v in MODEL_PATHS.items()}, "\n  missing (pre-train them first):", [k for k in ("gpt2_new", "modernbert", "t5", "bilstm") if k not in MODEL_PATHS])

def load_backbone_any(path):
    """like AL.load_backbone, but also accepts the OLD Phase 1 checkpoint (its model_config has no 'family' entry)"""
    ck = torch.load(path, map_location="cpu", weights_only=False); mc = dict(ck["model_config"]); mc.setdefault("family", "gpt2")
    full = AL.build_full(mc); full.load_state_dict(ck["model_state_dict"], strict=True); f = mc["family"]
    if f == "gpt2": inner, embed = full.transformer, full.transformer.wte
    elif f == "modernbert": inner, embed = full.model, full.model.embeddings.tok_embeddings
    elif f == "t5": inner, embed = full.encoder, full.shared
    elif f == "bilstm": inner, embed = full, full.emb
    else: raise ValueError(f)
    del ck; return AL.Backbone(f, inner, embed), mc["hidden_size"], AL.CAUSAL[f], mc["context_length"], f

def set_backbone_dropout(backbone, p):
    """turn on dropout inside the backbone for fine-tuning (works where the model has nn.Dropout modules, e.g. GPT-2 and T5; a no-op for the others). Returns how many modules were changed."""
    n = 0
    for mod in backbone.modules():
        if isinstance(mod, nn.Dropout): mod.p = p; n += 1
    return n

def truncate_layers(bb, n):
    """keep only the first n transformer layers (initialises a smaller student from the same Phase 1 weights); implemented for the GPT-2 family"""
    assert bb.family == "gpt2", "layer truncation is implemented for the GPT-2 family only"
    bb.inner.h = nn.ModuleList(list(bb.inner.h)[:n]); bb.inner.config.n_layer = n; return bb

def refresh_models(verbose=True):
    """re-scan /kaggle/input for Phase 1 models (for example after you attached another dataset to the running session)"""
    new = {k: v for k, v in discover_models().items() if k not in MODEL_PATHS}; MODEL_PATHS.update(new)
    if verbose and new: print("newly found Phase 1 models:", new)
    return sorted(MODEL_PATHS)

class MLClassifier(nn.Module):
    """Phase 1 backbone + a multi-label head. One window of <= L tokens in, one logit per statute out. Causal models (GPT-2) pool the last token, the others the mean of the tokens."""
    def __init__(self, backbone, hidden, causal, n_labels, dropout=0.1, pool="last"):
        super().__init__(); self.backbone, self.hidden, self.causal, self.n_labels, self.pool = backbone, hidden, causal, n_labels, pool
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, n_labels))
    def forward(self, ids, mask):
        T = int(mask.sum(1).max()); ids, mask = ids[:, :T], mask[:, :T]
        h = self.backbone(input_ids=ids, attention_mask=mask).last_hidden_state; mm = mask.unsqueeze(-1).to(h.dtype); mean = (h * mm).sum(1) / mm.sum(1).clamp(min=1)
        if self.causal:
            last = h[torch.arange(h.shape[0], device=h.device), mask.sum(1) - 1]; pooled = last if self.pool == "last" else 0.5 * (last + mean)
        else: pooled = mean
        return self.head(pooled)

# ---- windows over the stored tokens of a case (the whole case if <= 8,192 tokens, else its first 4,096 + last 4,096) ----
def window_spans(n, L, W, order):
    """up to W (start, end) spans of <= L tokens. order: 'head' (from the start), 'tail' (from the end), 'both' (head and tail windows interleaved, skipping a window already covered)"""
    nb = max(1, -(-n // L)); head = [(k * L, min(n, (k + 1) * L)) for k in range(nb)]; tail = [(max(0, n - (k + 1) * L), n - k * L) for k in range(nb)]
    if order == "head": sp = head
    elif order == "tail": sp = tail
    else:
        sp = []
        for a, b in zip(head, tail):
            for s_ in (a, b):
                if not any(s_[0] >= c[0] and s_[1] <= c[1] for c in sp): sp.append(s_)
    return sp[:W]
def build_batch(toks, idx, spans, L):
    ids = np.full((len(idx), L), PAD, dtype=np.int64); mask = np.zeros((len(idx), L), dtype=np.int64)
    for b, (i, (s, e)) in enumerate(zip(idx, spans)):
        seg = toks[i][s:e]; ids[b, :len(seg)] = seg; mask[b, :len(seg)] = 1
    return torch.from_numpy(ids), torch.from_numpy(mask)


In [ ]:
# ================= CELL 7 — ADAPTATION TECHNIQUES  [pure code]: full, partial (layer-wise freezing), LoRA, rsLoRA, LoRMA+, LoRMA-pi, IA3 =================
from transformers.pytorch_utils import Conv1D
class LoRADelta(nn.Module):                # additive:  dW x = (alpha/r) B A x   (rsLoRA: alpha/sqrt(r))
    def __init__(self, nx, nf, r, alpha, drop, rs=False):
        super().__init__()
        self.A = nn.Parameter(torch.empty(nx, r).uniform_(-1 / math.sqrt(nx), 1 / math.sqrt(nx))); self.B = nn.Parameter(torch.zeros(r, nf))
        self.scale = alpha / (math.sqrt(r) if rs else r); self.drop = nn.Dropout(drop)
    def forward(self, x): return (self.drop(x) @ self.A) @ self.B * self.scale
class LoRMAPlus(nn.Module):                # multiplicative, additive rank inflation: h' = (I + (alpha/r) B A) h ; B = 0 at init  (= W0 x + (alpha/r) B A (W0 x))
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d))); self.B = nn.Parameter(torch.zeros(d, r)); self.scale = alpha / r
    def forward(self, h): return h + ((h @ self.A.t()) @ self.B.t()) * self.scale
class LoRMAPi(nn.Module):                  # multiplicative, permutation-based rank inflation: h' = I_pi(BA) h, I_pi(BA) = I at init
    def __init__(self, d, r):
        super().__init__()
        B = torch.empty(d, r).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d)); B[:, 0] = 1.0; A = torch.zeros(r, d); A[0, 0] = 1.0
        self.B, self.A = nn.Parameter(B), nn.Parameter(A); i = torch.arange(d); self.register_buffer("idx", (i[None, :] - i[:, None]) % d)
    def matrix(self): return (self.B @ self.A).gather(1, self.idx)
    def forward(self, h): return h @ self.matrix().t().to(h.dtype)
class _Wrap:
    """expose the wrapped layer's weight / bias: some Hugging Face modules read them directly (T5's feed-forward dtype check does)"""
    @property
    def weight(self): return self.base.weight
    @property
    def bias(self): return self.base.bias
def proj_dims(m):
    return (m.in_features, m.out_features) if isinstance(m, nn.Linear) else tuple(m.weight.shape)      # HF Conv1D weight is [in, out]
class FusedQV(_Wrap, nn.Module):                  # wraps a FUSED q|k|v projection (GPT-2 Conv1D c_attn, ModernBERT Linear Wqkv); LoRA / LoRMA adapt q and v, IA3 rescales k and v
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__(); nx, nf = proj_dims(base); d = nf // 3; self.base, self.kind, self.d = base, kind, d
        if kind == "lora": self.q, self.v = LoRADelta(nx, d, r, alpha, drop, rs), LoRADelta(nx, d, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.q, self.v = LoRMAPlus(d, r, alpha), LoRMAPlus(d, r, alpha)
        elif kind == "lorma_pi": self.q, self.v = LoRMAPi(d, r), LoRMAPi(d, r)
        elif kind == "ia3": self.lk, self.lv = nn.Parameter(torch.ones(d)), nn.Parameter(torch.ones(d))
        else: raise ValueError(kind)
    def forward(self, x):
        out = self.base(x); d = self.d; q, k, v = out[..., :d], out[..., d:2 * d], out[..., 2 * d:]
        if self.kind == "lora": q, v = q + self.q(x), v + self.v(x)
        elif self.kind.startswith("lorma"): q, v = self.q(q), self.v(v)
        else: k, v = k * self.lk, v * self.lv
        return torch.cat([q, k, v], -1)
class IA3Input(_Wrap, nn.Module):                 # IA3 on the FFN: rescale the activation entering the down-projection
    def __init__(self, base): super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(proj_dims(base)[0]))
    def forward(self, x): return self.base(x * self.l)
class LinAdapt(_Wrap, nn.Module):                 # the same adapters for separate nn.Linear projections (T5's q, k, v)
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__(); self.base, self.kind = base, kind; di, do = base.in_features, base.out_features
        if kind == "lora": self.op = LoRADelta(di, do, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.op = LoRMAPlus(do, r, alpha)
        elif kind == "lorma_pi": self.op = LoRMAPi(do, r)
        elif kind == "ia3": self.l = nn.Parameter(torch.ones(do))
    def forward(self, x):
        y = self.base(x)
        if self.kind == "lora": return y + self.op(x)
        if self.kind == "ia3": return y * self.l
        return self.op(y)
def wrap_linear_targets(root, kind, names, **kw):
    n = 0
    for _, parent in list(root.named_modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and cname in names: setattr(parent, cname, LinAdapt(child, kind, **kw)); n += 1
    assert n > 0, f"no nn.Linear named {names} found in the backbone"
def freeze_lower(model, frac=0.5):
    """partial / layer-wise fine-tuning: freeze the embeddings and the lowest `frac` of the layers, train the rest"""
    inner, fam = model.backbone.inner, model.backbone.family
    for p in inner.parameters(): p.requires_grad = True
    if fam == "bilstm":
        n = inner.layers; k = int(n * frac)
        for p in inner.emb.parameters(): p.requires_grad = False
        for name, p in list(inner.fwd.named_parameters()) + list(inner.bwd.named_parameters()):
            if int(name.split("_l")[-1]) < k: p.requires_grad = False
        return
    if fam == "gpt2": layers, pre = list(inner.h), [inner.wte, inner.wpe]
    elif fam == "modernbert": layers, pre = list(inner.layers), [inner.embeddings]
    elif fam == "t5": layers, pre = list(inner.block), [inner.embed_tokens]
    else: raise ValueError(fam)
    k = int(len(layers) * frac)
    for mod in pre:
        for p in mod.parameters(): p.requires_grad = False
    for lay in layers[:k]:
        for p in lay.parameters(): p.requires_grad = False
METHODS = ("full", "partial", "lora", "rslora", "lorma_plus", "lorma_pi", "ia3")
ADAPTER_LR = {"lora": 5e-4, "rslora": 5e-4, "lorma_plus": 5e-4, "lorma_pi": 2e-4, "ia3": 3e-3}        # starting points; use lr_pilot(..., method=...) to check them
def apply_method(model, method, hp=None):
    assert method in METHODS, method; hp = hp or {}
    if method == "full":
        for p in model.backbone.parameters(): p.requires_grad = True
        return model
    if method == "partial": freeze_lower(model, hp.get("frac", 0.5)); return model
    fam, inner = model.backbone.family, model.backbone.inner
    for p in model.backbone.parameters(): p.requires_grad = False
    kind = {"lora": "lora", "rslora": "lora", "lorma_plus": "lorma_plus", "lorma_pi": "lorma_pi", "ia3": "ia3"}[method]; kw = dict(r=hp.get("r", 16), alpha=hp.get("alpha", 32), drop=hp.get("adapter_dropout", 0.05), rs=(method == "rslora"))
    if fam == "gpt2":
        for blk in inner.h:
            assert isinstance(blk.attn.c_attn, Conv1D), "unexpected c_attn type in this transformers version"; blk.attn.c_attn = FusedQV(blk.attn.c_attn, kind, **kw)
            if kind == "ia3": blk.mlp.c_proj = IA3Input(blk.mlp.c_proj)
    elif fam == "modernbert":
        for lay in inner.layers:
            assert isinstance(lay.attn.Wqkv, nn.Linear); lay.attn.Wqkv = FusedQV(lay.attn.Wqkv, kind, **kw)
            if kind == "ia3": lay.mlp.Wo = IA3Input(lay.mlp.Wo)
    elif fam == "t5":
        wrap_linear_targets(inner, kind, ("k", "v") if kind == "ia3" else ("q", "v"), **({} if kind == "ia3" else kw))
        if kind == "ia3":
            for _, parent in list(inner.named_modules()):
                if "DenseGatedActDense" in type(parent).__name__ or "DenseReluDense" in type(parent).__name__: parent.wo = IA3Input(parent.wo)
    else: raise NotImplementedError(f"'{method}' needs attention/linear layers: the {fam} model supports only 'full' and 'partial'")
    return model


In [ ]:
# ================= CELL 8 — TRAINER (multi-label)  [pure code; needs torch]: any adaptation method, optional distillation, three losses, head/tail windows, EMA, window aggregation, checkpoint saving =================
import contextlib
USE_BF16 = device.type == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8
def amp_ctx():
    if not AMP: return contextlib.nullcontext()
    return torch.autocast("cuda", dtype=torch.bfloat16 if USE_BF16 else torch.float16)
DEFAULT_LR = {"gpt2": 3e-5, "modernbert": 5e-5, "t5": 5e-5, "bilstm": 5e-4}      # gpt2: the best of 3e-5 / 1e-4 / 3e-4 in the Phase 4 pilot; the others are starting points (lr_pilot_ml below checks them)
AGGS = ("max", "mean", "lse")                                                     # how the scores of one case's windows are combined: strongest window, average, or log-mean-exp (in between)
PEFT_METHODS = ("lora", "rslora", "lorma_plus", "lorma_pi", "ia3")
CKPT = Path(CKPT_DIR); CKPT.mkdir(parents=True, exist_ok=True)
def ckpt_files(name): return CKPT / f"{name}.pt", CKPT / f"{name}.json"
def ckpt_exists(name): return ckpt_files(name)[0].exists() and ckpt_files(name)[1].exists()

@torch.no_grad()
def predict_ml(model, toks, L, W, order, tok_budget=16384):
    """scores of every window of every case, combined three ways: {'max','mean','lse'} -> [cases, statutes] logits"""
    model.eval(); n = len(toks); spans = [window_spans(len(t), L, W, order) for t in toks]; mb = max(1, tok_budget // L)
    mx = np.full((n, V), -np.inf, np.float32); sm = np.zeros((n, V), np.float32); cnt = np.zeros(n, np.float32); lse = np.full((n, V), -np.inf, np.float32)
    for k in range(W):
        idx = [i for i in range(n) if len(spans[i]) > k]
        if not idx: continue
        idx = sorted(idx, key=lambda i: -(spans[i][k][1] - spans[i][k][0]))
        for s in range(0, len(idx), mb):
            sel = idx[s:s + mb]; ids, mask = build_batch(toks, sel, [spans[i][k] for i in sel], L)
            with amp_ctx(): lg = model(ids.to(device), mask.to(device)).float().cpu().numpy()
            mx[sel] = np.maximum(mx[sel], lg); sm[sel] += lg; cnt[sel] += 1; lse[sel] = np.logaddexp(lse[sel], lg)
    return {"max": mx, "mean": sm / cnt[:, None], "lse": lse - np.log(cnt)[:, None]}
class EMA:
    """exponential moving average of the weights; evaluated instead of the raw weights (smoother, usually better at the end of fine-tuning)"""
    def __init__(self, model, decay):
        self.params = [p for p in model.parameters() if p.requires_grad]; self.shadow = [p.detach().clone() for p in self.params]; self.decay = decay; self.backup = None
    @torch.no_grad()
    def update(self, step):
        d = min(self.decay, (1 + step) / (10 + step)); torch._foreach_mul_(self.shadow, d); torch._foreach_add_(self.shadow, [p.detach() for p in self.params], alpha=1 - d)
    @torch.no_grad()
    def apply(self):
        self.backup = [p.detach().clone() for p in self.params]
        for p, s in zip(self.params, self.shadow): p.copy_(s)
    @torch.no_grad()
    def restore(self):
        for p, b in zip(self.params, self.backup): p.copy_(b)
        self.backup = None


def run_experiment(name, model_key, method="full", hp=None, n_layers=None, kd=None, L=512, W=4, order="both", loss="bce", epochs=3.0, max_minutes=None, lr=None, head_lr=1e-3, bs=16, wd=0.01, warmup=0.06,
                   p_first=0.5, ema_decay=0.999, pool="last", dropout=0.1, seed=SEED, max_train_docs=None, dev_cap=None, test_cap=None, evals=4, write=True, save_model=SAVE_MODELS, control=False, exp_id=None):
    """method: full | partial | lora | rslora | lorma_plus | lorma_pi | ia3 ; kd=dict(path=..., T=2.0, alpha=0.5) distils from make_teacher_ml() ; n_layers=6 = half-depth GPT-2 student ; order: head | tail | both ;
       loss: bce | bce_posw | asl ; control=True marks an ablation that is never one of the six submissions"""
    t_start = time.time(); seed_all(seed); rng = np.random.default_rng(seed)
    assert model_key in MODEL_PATHS, f"'{model_key}' is not available: pre-train it first (found: {list(MODEL_PATHS)})"
    bb, hidden, causal, ctx, fam = load_backbone_any(MODEL_PATHS[model_key]); L = min(L, ctx)
    if n_layers: truncate_layers(bb, n_layers)
    lr = lr or (DEFAULT_LR[fam] if method in ("full", "partial") else ADAPTER_LR[method]); technique = "kd" if kd else method
    model = MLClassifier(bb, hidden, causal, V, dropout, pool); apply_method(model, method, hp); model = model.to(device)
    n_params = sum(p.numel() for p in model.parameters()); n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tr, dv, te = DATA["train"], DATA["dev"], DATA["test"]
    tr_idx = np.arange(len(tr["toks"])) if not max_train_docs else rng.choice(len(tr["toks"]), min(max_train_docs, len(tr["toks"])), replace=False)
    dv_idx = np.arange(len(dv["toks"])) if not dev_cap else rng.choice(len(dv["toks"]), min(dev_cap, len(dv["toks"])), replace=False); te_idx = np.arange(len(te["toks"])) if not test_cap else np.arange(min(test_cap, len(te["toks"])))
    toks_tr = tr["toks"]; dv_toks = [dv["toks"][i] for i in dv_idx]; te_toks = [te["toks"][i] for i in te_idx]; Ydv = YDV[dv_idx]
    sub_n = min(1000, len(dv_idx)); sub = rng.choice(len(dv_idx), sub_n, replace=False); sub_toks = [dv_toks[i] for i in sub]; Ysub = Ydv[sub]
    spans_tr = [window_spans(len(t), L, W, order) for t in toks_tr]
    pos = YTR[tr_idx].sum(0).astype(np.float32); pw = torch.tensor(np.clip(np.sqrt((len(tr_idx) - pos) / np.maximum(pos, 1)), 1, 30), dtype=torch.float32, device=device)
    def loss_fn(lg, y):
        if loss == "bce": return F.binary_cross_entropy_with_logits(lg, y)
        if loss == "bce_posw": return F.binary_cross_entropy_with_logits(lg, y, pos_weight=pw)
        if loss == "asl":
            p = torch.sigmoid(lg); pn = (p - 0.05).clamp(min=0); return -(y * torch.log(p.clamp(min=1e-8)) + (1 - y) * pn ** 4 * torch.log((1 - pn).clamp(min=1e-8))).mean()
        raise ValueError(loss)
    tl = None
    if kd: tl = np.load(kd["path"]).astype(np.float32); assert tl.shape == (len(toks_tr), V), f"teacher logits {tl.shape} do not match the training set {(len(toks_tr), V)}: run make_teacher_ml() on the full training set"
    mb = max(1, 8192 // L); accum = max(1, math.ceil(bs / mb)); docs_per_step = mb * accum
    bb_train = [p for p in model.backbone.parameters() if p.requires_grad]; groups = ([{"params": bb_train, "lr": lr}] if bb_train else []) + [{"params": [p for n, p in model.named_parameters() if not n.startswith("backbone.")], "lr": head_lr}]
    opt = torch.optim.AdamW(groups, weight_decay=wd); base_lrs = [g["lr"] for g in groups]; scaler = torch.amp.GradScaler("cuda", enabled=AMP and not USE_BF16); ema = EMA(model, ema_decay)
    def micro(sel):
        sps = [spans_tr[i][0] if (len(spans_tr[i]) == 1 or rng.random() < p_first) else spans_tr[i][int(rng.integers(0, len(spans_tr[i])))] for i in sel]       # the first window with probability p_first, else any window
        ids, mask = build_batch(toks_tr, sel, sps, L); y = torch.from_numpy(YTR[sel].astype(np.float32)).to(device)
        with amp_ctx(): lg = model(ids.to(device), mask.to(device))
        lg = lg.float(); hard = loss_fn(lg, y)
        if tl is None: return hard
        T_, a_ = kd.get("T", 2.0), kd.get("alpha", 0.5); q = torch.sigmoid(torch.from_numpy(tl[sel]).to(device) / T_)
        return a_ * hard + (1 - a_) * F.binary_cross_entropy_with_logits(lg / T_, q) * T_ * T_
    model.train(); times = []
    for j in range(4):                                                            # throughput probe (forward + backward only); the first is warm-up
        t0 = time.time(); loss_ = micro(rng.choice(tr_idx, mb)); scaler.scale(loss_).backward(); opt.zero_grad(set_to_none=True)
        if device.type == "cuda": torch.cuda.synchronize()
        times.append(time.time() - t0)
    micro_s = float(np.mean(times[1:])); sec_step = micro_s * accum; per_doc_fwd = micro_s / mb / 3.0
    steps = max(10, int(math.ceil(epochs * len(tr_idx) / docs_per_step))); eval_s = ((len(dv_toks) + len(te_toks)) * W + evals * sub_n) * per_doc_fwd
    if max_minutes: steps = min(steps, max(10, int((max_minutes * 60 * 0.92 - eval_s) / sec_step)))
    print(f"[{name}] {fam} | technique {technique}{' (method ' + method + ')' if kd else ''} | L={L} W={W} order={order} loss={loss} | {n_params/1e6:.0f}M params, {n_train/1e6:.2f}M trained ({n_train/n_params:.1%}) | lr {lr:g} (head {head_lr:g}) | {docs_per_step} cases/step\n"
          f"   plan: {steps} steps = {steps*docs_per_step} cases = {steps*docs_per_step/len(tr_idx):.2f} epochs | {sec_step:.2f}s/step -> ~{steps*sec_step/60:.0f} min training + ~{eval_s/60:.1f} min inference")
    perm, ptr, evals_at, run_loss, hist = rng.permutation(tr_idx), 0, {int(round(steps * (i + 1) / evals)) for i in range(evals)}, 0.0, []
    for step in range(1, steps + 1):
        w = min(step / max(1, int(warmup * steps)), (steps - step + 1) / max(1, steps - int(warmup * steps)), 1.0)
        for g, b in zip(opt.param_groups, base_lrs): g["lr"] = b * w
        for _ in range(accum):
            if ptr + mb > len(perm): perm, ptr = rng.permutation(tr_idx), 0
            loss_ = micro(perm[ptr:ptr + mb]); ptr += mb; scaler.scale(loss_ / accum).backward(); run_loss += loss_.item() / accum
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); ema.update(step)
        if step % max(1, steps // 12) == 0: print(f"   step {step}/{steps} | train loss {run_loss/max(1, steps//12):.4f} | {(time.time()-t_start)/60:.1f} min", flush=True); run_loss = 0.0
        if step in evals_at:                                                       # monitoring only (EMA weights, first window, dev subset): threshold-free macro average precision
            ema.apply(); o = predict_ml(model, sub_toks, L, 1, order); ema.restore(); model.train(); ap_ = macro_ap(Ysub, sigmoid(o["max"])); hist.append((step, round(ap_, 4))); print(f"   [dev subset {sub_n}, EMA, first window] step {step}: macro AP {ap_:.4f}", flush=True)
    ema.apply()                                                                    # final model = EMA weights
    Od, Ot = predict_ml(model, dv_toks, L, W, order), predict_ml(model, te_toks, L, W, order)
    aps = {a: macro_ap(Ydv, sigmoid(Od[a])) for a in AGGS}; agg = max(AGGS, key=lambda a: aps[a]); Sd = sigmoid(Od[agg]); sel = select_policy(Sd, Ydv, card_dev); Pd = apply_policy(sel, Sd, card_dev); m = f1_scores(Ydv, Pd)
    Pt = apply_policy(sel, sigmoid(Ot[agg]), card_dev); one = {a: macro_ap(Ydv, sigmoid(predict_ml(model, dv_toks, L, 1, order)[a])) for a in ("max",)} if W > 1 else {"max": aps[agg]}
    res = dict(name=name, model=model_key, family=fam, method=method, technique=technique, kd=bool(kd), n_layers=n_layers or 0, control=bool(control), L=L, W=W, order=order, loss=loss, params_M=round(n_params / 1e6, 1), trainable_M=round(n_train / 1e6, 3),
               lr=lr, epochs=round(steps * docs_per_step / len(tr_idx), 2), steps=steps, agg=agg, policy=sel["policy"], exp_id=exp_id or "", dev_macro_f1=sel["scores"][sel["policy"]], dev_macro_f1_present=m["macro_f1_present"], dev_macro_f1_universe=m["macro_f1_universe"], dev_macro_f1_universe_zd1=m["macro_f1_universe_zd1"], dev_micro_f1=m["micro_f1"], dev_subset_acc=m["subset_acc"],
               dev_macro_ap=aps[agg], dev_macro_ap_first_window=one["max"], tau=sel["tau"], policy_scores=json.dumps({k: round(v, 4) for k, v in sel["scores"].items()}), test_mean_labels=float(Pt.sum(1).mean()), dev_mean_labels=float(Pd.sum(1).mean()),
               dev_cases=len(dv_toks), minutes=round((time.time() - t_start) / 60, 1), history=str(hist))
    print(f"[{name}] DEV: macro-F1 [{MACRO_CONVENTION}] {res['dev_macro_f1']:.4f} (policy {sel['policy']}; all policies {res['policy_scores']}) | present-in-val {m['macro_f1_present']:.4f} | universe {m['macro_f1_universe']:.4f} (zd1 {m['macro_f1_universe_zd1']:.4f}) | micro-F1 {m['micro_f1']:.4f} | subset acc {m['subset_acc']:.4f} | macro AP {aps[agg]:.4f} (first window only {one['max']:.4f}; aggregation '{agg}') | {res['minutes']} min\n"
          f"   labels per case: dev true {card_dev:.2f}, dev predicted {Pd.sum(1).mean():.2f}, test predicted {Pt.sum(1).mean():.2f} | TF-IDF baseline macro-F1 {globals().get('BASELINE', {}).get('macro_f1', float('nan')):.4f}")
    if write:
        rd = WORK / "runs" / name; rd.mkdir(exist_ok=True); np.save(rd / "dev_logits.npy", Od[agg]); np.save(rd / "test_logits.npy", Ot[agg]); json.dump(res, open(rd / "result.json", "w"), indent=1)
        np.save(rd / "dev_pred.npy", Pd.astype(np.uint8)); np.save(rd / "dev_idx.npy", dv_idx)
        if save_model:                                                             # adapters (PEFT) are saved alone: the frozen base is the Phase 1 checkpoint; every other method saves the whole fine-tuned model
            peft = method in PEFT_METHODS; trainable = {n for n, p in model.named_parameters() if p.requires_grad}; state = {k: v.detach().cpu() for k, v in model.state_dict().items() if (not peft or k in trainable)}
            pth, meta_p = ckpt_files(name); torch.save({"state": state, "peft_only": peft}, pth); json.dump(dict(model_key=model_key, method=method, hp=hp, n_layers=n_layers, pool=pool, L=L, W=W, order=order, agg=agg, labels=[int(x) for x in LABELS], dev_macro_f1=res["dev_macro_f1"]), open(meta_p, "w"))
            print(f"   checkpoint saved: {pth} ({pth.stat().st_size/2**20:.0f} MiB{', adapters + head only' if peft else ''})")
        rids = [te["ids"][i] for i in te_idx]; write_submission_ml(SUBDIR / f"{name}.json", rids, lists_from(Pt, Ot[agg])); validate_submission_ml(SUBDIR / f"{name}.json", rids)
        row = pd.DataFrame([res]); led = pd.concat([pd.read_csv(LEDGER).query('name != @name'), row]) if LEDGER.exists() else row; led.to_csv(LEDGER, index=False)
        print(f"   saved {SUBDIR / (name + '.json')} ({len(rids)} cases, format validated) and the ledger row")
    del model, opt, bb, ema; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return res

def load_finetuned_ml(run_name):
    """rebuild a fine-tuned model from its checkpoint (the Phase 1 weights come from MODEL_PATHS; PEFT checkpoints only hold the adapters + head)"""
    pth, meta_p = ckpt_files(run_name); meta = json.load(open(meta_p)); ck = torch.load(pth, map_location="cpu", weights_only=False); assert meta["labels"] == [int(x) for x in LABELS], "this checkpoint was trained on a different label universe"
    bb, hidden, causal, ctx, fam = load_backbone_any(MODEL_PATHS[meta["model_key"]])
    if meta.get("n_layers"): truncate_layers(bb, meta["n_layers"])
    model = MLClassifier(bb, hidden, causal, V, 0.0, meta.get("pool", "last")); apply_method(model, meta["method"], meta.get("hp")); missing, unexpected = model.load_state_dict(ck["state"], strict=False)
    trainable = {n for n, p in model.named_parameters() if p.requires_grad}; assert not unexpected and (not ck.get("peft_only") or all(k not in trainable for k in missing)), f"checkpoint {pth} does not match the model: unexpected {unexpected[:3]}, missing trainable {[k for k in missing if k in trainable][:3]}"
    return model.to(device).eval(), meta

@torch.no_grad()
def make_teacher_ml(run_names, out=None):
    """teacher for distillation: the mean logits (all windows, the run's own aggregation) of finished runs on EVERY training case. One inference pass over the training set per run."""
    tr = DATA["train"]; logits = []
    for name in run_names:
        model, meta = load_finetuned_ml(name); t0 = time.time(); o = predict_ml(model, tr["toks"], meta["L"], meta["W"], meta["order"])[meta["agg"]]; logits.append(o)
        print(f"teacher {name}: training-set macro AP {macro_ap(YTR, sigmoid(o)):.3f} (high is expected: it was trained on these cases) | {(time.time()-t0)/60:.1f} min"); del model; gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()
    tm = np.mean(logits, 0).astype(np.float32); path = Path(out) if out else WORK / "teacher_logits.npy"; np.save(path, tm); print(f"teacher = mean logits of {run_names} -> {path} {tm.shape}"); return str(path)

def lr_pilot_ml(model_key, grid=(1.5e-5, 3e-5, 1e-4), minutes=10, L=512, **kw):
    """short runs on a subset; picks the learning rate by dev macro average precision (threshold-free). Differences below about 0.01 are noise: prefer the middle value then."""
    out = {}
    for lr_ in grid:
        r = run_experiment(f"pilot_{model_key}_{lr_:g}", model_key, L=L, W=1, epochs=99, max_minutes=minutes, lr=lr_, max_train_docs=10000, dev_cap=1000, test_cap=20, evals=1, write=False, ema_decay=0.99, **kw); out[lr_] = r["dev_macro_ap"]
    best = max(out, key=out.get); print("\nLR pilot (dev macro AP):", {f"{k:g}": round(v, 4) for k, v in out.items()}, "-> best", f"{best:g}"); return best

# ---- run only what can run: experiments whose Phase 1 model is missing are skipped (with a message) ----
def run_or_skip(name, model_key, **kw):
    if model_key not in MODEL_PATHS: refresh_models(verbose=False)
    if model_key not in MODEL_PATHS:
        print(f"SKIPPED {name}: the Phase 1 '{model_key}' model is not available yet (found: {sorted(MODEL_PATHS)}). Attach it or finish its pre-training, then re-run this cell."); return None
    return run_experiment(name, model_key, **kw)
def pilot_or_skip(model_key, default_lr, **kw):
    if model_key not in MODEL_PATHS: refresh_models(verbose=False)
    if model_key not in MODEL_PATHS:
        print(f"SKIPPED the learning-rate pilot for '{model_key}' (model not available); the default {default_lr:g} is used if you run its experiment later."); return default_lr
    return lr_pilot_ml(model_key, **kw)
def show_plan():
    plan = [("E1_gpt2_full", "gpt2_new", "decoder: full fine-tuning"), ("E6_enc_full", "modernbert", "encoder: full fine-tuning"), ("E2_enc_lorma", "modernbert", "encoder: LoRMA+ (compulsory)"), ("E3_t5_*", "t5", "enc-dec: LoRA (full fine-tuning if the probe says so)"),
            ("E4_bilstm_partial", "bilstm", "recurrent: partial fine-tuning"), ("E5_gpt2_6L_kd", "gpt2_new", "decoder student: knowledge distillation (needs the saved checkpoints of E1 and/or E6)")]
    refresh_models(verbose=False); done = set(pd.read_csv(LEDGER).name) if LEDGER.exists() else set()
    print("Phase 1 models available:", sorted(MODEL_PATHS), "| MISSING:", [k for k in ("gpt2_new", "modernbert", "t5", "bilstm") if k not in MODEL_PATHS] or "none")
    print(f"{'experiment':20s} {'model':11s} {'what':64s} status")
    for n, m, tech in plan:
        teach = ckpt_exists("E1_gpt2_full") or ckpt_exists("E6_enc_full")
        st = "done" if (n in done or (n.endswith("*") and any(d.startswith(n[:-1]) for d in done))) else ("SKIPPED: model missing" if m not in MODEL_PATHS else ("will run once E1 or E6 has finished" if (n == "E5_gpt2_6L_kd" and not teach) else "will run"))
        print(f"{n:20s} {m:11s} {tech:64s} {st}")

# ---- ensemble of finished runs: the mean of validation-standardised logits; the number of models is chosen on validation, preferring MORE models when within 0.002 ----
def margins_of(n):
    rd = WORK / "runs" / n; d = np.load(rd / "dev_logits.npy"); t_ = np.load(rd / "test_logits.npy"); s = d.std() + 1e-9; return d / s, t_ / s
def build_ensemble(order_names):
    M = {n: margins_of(n) for n in order_names}; sc = []
    for k in range(1, len(order_names) + 1):
        z = np.mean([M[n][0] for n in order_names[:k]], 0); s_ = select_policy(sigmoid(z), YDV, card_dev); sc.append(s_["scores"][s_["policy"]])
    k_best = max(k for k, v in enumerate(sc, 1) if v >= max(sc) - 0.002); names = order_names[:k_best]
    z_dev = np.mean([M[n][0] for n in names], 0); z_te = np.mean([M[n][1] for n in names], 0); sel = select_policy(sigmoid(z_dev), YDV, card_dev)
    return dict(names=names, z_dev=z_dev, z_te=z_te, sel=sel, score_by_k=dict(zip(order_names, [round(v, 4) for v in sc])))


In [ ]:
# ================= RESTORE PART 1's RESULTS  [NEEDS the backup dataset attached (Add Input > Your datasets > cs781-phase5-lsi-results)]. Copies the checkpoints, run results, submissions, ledger and window order back, so E5 finds its teachers and the final cells see all experiments. =================
# Copies the backed-up checkpoints, run results, submissions and ledger back into this session, so the experiment cells that already finished are not repeated and E5 (distillation) can find its teachers.
mfs = [m for m in glob.glob("/kaggle/input/**/MANIFEST.json", recursive=True) if re.search(r"phase5|lsi", m, re.I)]
assert mfs, "attach the backup dataset (cs781-phase5-lsi-results) first"
src = Path(mfs[-1]).parent; man = json.load(open(mfs[-1])); bad = []
for name, meta in man["files"].items():
    f = src / name
    if not f.exists() or f.stat().st_size != meta["bytes"]: bad.append(name); continue
    run, fn = name.split("__", 1)
    dst = SUBDIR / fn if run == "submission" else CKPT / fn if run == "ckpt" else (WORK / fn if fn in ("ledger.csv", "order.json") else Path("/kaggle/working") / fn) if run == "top" else WORK / "runs" / run / fn
    dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(f, dst)
print(f"restored {len(man['files']) - len(bad)} files for runs {man.get('runs')} from {src}" + (f" | MISSING or wrong size: {bad}" if bad else " | all sizes match the manifest"))
if (WORK / "order.json").exists(): ORDER = json.load(open(WORK / "order.json"))["order"]; print("window order restored from Part 1:", ORDER)


In [ ]:
# ===== CELL 10 — RESILIENCE HELPER + SHARED SETTINGS  [pure code; run it before the experiment cells]: a failing experiment prints its traceback and the notebook carries on, so one crash cannot stop 'Save & Run All' before the later cells =====
import traceback
def safe(label, fn):
    try: return fn()
    except Exception:
        print(f"{label} FAILED, continuing with the next cell:"); traceback.print_exc(); gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return None
ORDER = globals().get("ORDER") or (json.load(open(WORK / "order.json"))["order"] if (WORK / "order.json").exists() else "both")      # head | tail | both: set by the region test / the window study, or restored from order.json; "both" if none of them ran
W_MAIN = 4                                             # windows of 512 tokens per case in every experiment (2,048 tokens: the same budget for all experiments)
CHANCE_AP = float(np.mean(YDV[:, YDV.sum(0) > 0].mean(0)))      # macro average precision of a model with no signal (the mean statute prevalence)
print(f"window order {ORDER} | {W_MAIN} windows of 512 tokens | chance-level macro AP {CHANCE_AP:.3f}")


In [ ]:
# ================= CELL 11 — EXPERIMENT E4: RECURRENT (BiLSTM), PARTIAL FINE-TUNING  [NEEDS the BiLSTM Phase 1 checkpoint + a GPU; pilot ~30 min + run ~90 min] =================
# The embeddings and the lowest LSTM layer are frozen, the rest is trained. The BiLSTM has no attention layers, so adapters do not apply to it. Part 2 uses 512-token windows and 2 epochs so that the BiLSTM (about 9k tokens/s) fits the session; the combined notebook uses 1,024 tokens and 3 epochs.
LR_RNN = safe("BiLSTM pilot", lambda: pilot_or_skip("bilstm", DEFAULT_LR["bilstm"], grid=(2e-4, 5e-4, 1e-3), minutes=6, L=512, method="partial", hp=dict(frac=0.5))) or DEFAULT_LR["bilstm"]
E4 = safe("E4", lambda: run_or_skip("E4_bilstm_partial", "bilstm", exp_id="E4", method="partial", hp=dict(frac=0.5), L=512, W=W_MAIN, order=ORDER, epochs=2, lr=LR_RNN, save_model=True))


In [ ]:
# ================= CELL 12 — EXPERIMENT E5: KNOWLEDGE DISTILLATION  [NEEDS the saved checkpoints of E1 and/or E6 (the teachers) + the GPT-2 Phase 1 checkpoint + a GPU; teacher scoring ~40 min per teacher, student ~60 min] =================
# Teacher = the mean logits of E1 (decoder) and E6 (encoder) on every training case (one inference pass each). Student = the first 6 layers of the Phase 1 GPT-2. Loss = alpha * BCE(hard labels) + (1 - alpha) * T^2 * BCE(student/T, sigmoid(teacher/T)).
# E5c is the SAME student without distillation, so the benefit of distillation is measured (an ablation, not counted). If only one teacher exists it is used alone; if none exists the cell is skipped.
TEACH = [n for n in ("E1_gpt2_full", "E6_enc_full") if ckpt_exists(n)]
if not TEACH: print("SKIPPED E5 (distillation): no teacher checkpoint yet: run E1 and/or E6 first.")
elif "gpt2_new" not in MODEL_PATHS and "gpt2_new" not in refresh_models(verbose=False): print("SKIPPED E5 (distillation): the student is a half-depth GPT-2 and the GPT-2 Phase 1 checkpoint is not available.")
else:
    print("teachers:", TEACH, "(single teacher)" if len(TEACH) == 1 else "(ensemble)"); TEACHER = make_teacher_ml(TEACH)
    E5 = safe("E5", lambda: run_or_skip("E5_gpt2_6L_kd", "gpt2_new", exp_id="E5", method="full", n_layers=6, kd=dict(path=TEACHER, T=2.0, alpha=0.5), L=512, W=W_MAIN, order=ORDER, epochs=3, lr=5e-5, save_model=True))
    E5c = safe("E5c", lambda: run_or_skip("E5c_gpt2_6L_nokd", "gpt2_new", method="full", n_layers=6, L=512, W=W_MAIN, order=ORDER, epochs=3, lr=5e-5, control=True, save_model=False))


In [ ]:
# ================= CELL 13 — OPTIONAL EXTRAS  [NEED the same checkpoints as the experiment they name]. Hyperparameter variants are NOT experiments: they are ablations (control=True) and are never counted. =================
# (a) FALLBACK for technique coverage: if no LoRA run exists (E3 skipped or full fine-tuning), LoRA on GPT-2 is a distinct (model, technique) experiment.
led_now = pd.read_csv(LEDGER) if LEDGER.exists() else pd.DataFrame(columns=["name", "technique"])
if "lora" not in set(led_now.get("technique", [])): E7 = safe("E7", lambda: run_or_skip("E7_gpt2_lora", "gpt2_new", exp_id="E7", method="lora", hp=dict(r=16, alpha=32), L=512, W=W_MAIN, order=ORDER, epochs=4, lr=ADAPTER_LR["lora"], save_model=True))
else: print("a LoRA experiment already exists: E7 not needed")
# (b) ablations, uncomment what you want (each ~60-90 min). A different loss or seed changes a hyperparameter, not the experiment:
# A1 = run_or_skip("A1_gpt2_asl", "gpt2_new", method="full", L=512, W=W_MAIN, order=ORDER, epochs=3, lr=3e-5, loss="asl", control=True)       # asymmetric loss for the long tail of rare statutes
# A2 = run_or_skip("A2_gpt2_posw", "gpt2_new", method="full", L=512, W=W_MAIN, order=ORDER, epochs=3, lr=3e-5, loss="bce_posw", control=True)  # positive-weighted BCE
# A3 = run_or_skip("A3_gpt2_seed7", "gpt2_new", method="full", L=512, W=W_MAIN, order=ORDER, epochs=3, lr=3e-5, seed=7, control=True)          # a second seed


In [ ]:
# ================= CELL 14 — SALVAGE + ENSEMBLE + prediction.json  [RUNS after any experiment; trains nothing]. Reads the runs on disk, optionally scores MORE WINDOWS with saved checkpoints, builds the best ensemble, writes the submission. =================
import shutil
WINDOWS_TRY, ADOPT_GAIN, OUT_DIR = (), 0.003, Path("/kaggle/working")        # windows of 512 tokens scored with each saved model (the cache holds up to 8 per case; set (8,) to try it, about 45 min per model); adopted only if validation macro-F1 improves by ADOPT_GAIN
RUNS = WORK / "runs"; rows = []
for rd in sorted(RUNS.glob("*")):
    if rd.name.startswith(("pilot", "probe", "smoke")) or not (rd / "result.json").exists() or not (rd / "dev_logits.npy").exists(): continue
    r = json.load(open(rd / "result.json")); r["has_ckpt"] = ckpt_exists(rd.name); rows.append(r)
assert rows, f"no finished experiment under {RUNS}: run at least one experiment cell first"
inv = pd.DataFrame(rows); inv["control"] = inv.control.astype(bool); inv["full_dev"] = inv.dev_cases == len(YDV); inv["usable"] = (inv.dev_macro_ap >= CHANCE_AP + 0.02) & ~inv.control & inv.full_dev      # counted experiments, scored on the whole validation split, with a signal
print("RUNS ON DISK:"); print(inv[["name", "model", "technique", "control", "epochs", "dev_macro_f1", "dev_macro_f1_present", "dev_macro_f1_universe", "dev_micro_f1", "dev_macro_ap", "policy", "has_ckpt", "usable"]].round(4).to_string(index=False))
for _, r_ in inv[~inv.usable].iterrows(): print(f"excluded from the ensemble: {r_['name']} ({'ablation / control run' if r_['control'] else 'scored on a validation subset only' if not r_['full_dev'] else f'macro AP not above chance {CHANCE_AP:.3f} + 0.02'})")
rids = DATA["test"]["ids"]
for _, r in inv[inv.has_ckpt & inv.usable].iterrows():                         # more windows with the saved checkpoints
    model, meta = load_finetuned_ml(r["name"])
    for Wt in WINDOWS_TRY:
        if Wt <= meta["W"]: continue
        t0 = time.time(); Od = predict_ml(model, DATA["dev"]["toks"], meta["L"], Wt, meta["order"]); aps = {a: macro_ap(YDV, sigmoid(Od[a])) for a in AGGS}; agg = max(AGGS, key=lambda a: aps[a]); Sd = sigmoid(Od[agg]); sel = select_policy(Sd, YDV, card_dev); sc = sel["scores"][sel["policy"]]
        print(f"\n{r['name']}: {meta['W']} -> {Wt} windows: validation macro-F1 {r['dev_macro_f1']:.4f} -> {sc:.4f} (aggregation {agg}) | {(time.time()-t0)/60:.1f} min")
        if sc >= r["dev_macro_f1"] + ADOPT_GAIN:
            Ot = predict_ml(model, DATA["test"]["toks"], meta["L"], Wt, meta["order"]); Pd = apply_policy(sel, Sd, card_dev); Pt = apply_policy(sel, sigmoid(Ot[agg]), card_dev); m = evaluate(YDV, Pd); new = f"{r['name']}_W{Wt}"; nd = RUNS / new; nd.mkdir(exist_ok=True)
            res = {k: v for k, v in r.items() if k not in ("has_ckpt", "usable")}; res.update(name=new, W=Wt, agg=agg, policy=sel["policy"], dev_macro_f1=sc, dev_macro_f1_present=m["macro_f1_present"], dev_macro_f1_universe=m["macro_f1_universe"], dev_macro_f1_universe_zd1=m["macro_f1_universe_zd1"], dev_micro_f1=m["micro_f1"], dev_subset_acc=m["subset_acc"], dev_macro_ap=aps[agg])
            np.save(nd / "dev_logits.npy", Od[agg]); np.save(nd / "test_logits.npy", Ot[agg]); json.dump(res, open(nd / "result.json", "w"), indent=1); write_submission_ml(SUBDIR / f"{new}.json", rids, lists_from(Pt, Ot[agg])); validate_submission_ml(SUBDIR / f"{new}.json", rids)
            led = pd.concat([pd.read_csv(LEDGER).query("name != @new"), pd.DataFrame([res])]); led.to_csv(LEDGER, index=False); rows.append({**res, "has_ckpt": False}); print(f"   ADOPTED as '{new}' (same experiment, same checkpoint, more windows at inference)")
        else: print(f"   not adopted (needs +{ADOPT_GAIN})")
    del model; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
inv = pd.DataFrame(rows); inv["control"] = inv.control.astype(bool); inv["usable"] = (inv.dev_macro_ap >= CHANCE_AP + 0.02) & ~inv.control & (inv.dev_cases == len(YDV)); cand = inv[inv.usable].sort_values("dev_macro_f1", ascending=False).reset_index(drop=True); assert len(cand), "no usable run"
ens = build_ensemble(list(cand.name)); best_single = cand.iloc[0]["name"]; best_sc = float(cand.iloc[0]["dev_macro_f1"]); ens_sc = ens["sel"]["scores"][ens["sel"]["policy"]]; use_ens = len(ens["names"]) > 1 and ens_sc >= best_sc + ADOPT_GAIN
print("\nensemble validation macro-F1 by number of models (best first):", ens["score_by_k"], f"-> using {ens['names']} (policy {ens['sel']['policy']})")
if use_ens: z_dev, z_te, sel_f, name_f = ens["z_dev"], ens["z_te"], ens["sel"], f"ensemble of {ens['names']}"
else: zd, zt = margins_of(best_single); z_dev, z_te, sel_f, name_f = zd, zt, select_policy(sigmoid(zd), YDV, card_dev), best_single
Pd = apply_policy(sel_f, sigmoid(z_dev), card_dev); Pt = apply_policy(sel_f, sigmoid(z_te), card_dev); m = evaluate(YDV, Pd); rng = np.random.default_rng(0)
boot = [f1_scores(YDV[i], Pd[i])["macro_f1"] for i in (rng.integers(0, len(YDV), len(YDV)) for _ in range(200))]; lo, hi = np.percentile(boot, [2.5, 97.5])
write_submission_ml(OUT_DIR / "prediction.json", rids, lists_from(Pt, z_te)); validate_submission_ml(OUT_DIR / "prediction.json", rids)
zb, tb = margins_of(best_single); write_submission_ml(OUT_DIR / "prediction_best_single.json", rids, lists_from(apply_policy(select_policy(sigmoid(zb), YDV, card_dev), sigmoid(tb), card_dev), tb)); validate_submission_ml(OUT_DIR / "prediction_best_single.json", rids)
print(f"\n=== prediction.json = {name_f} (chosen on validation only; thresholds tuned on validation only; policy {sel_f['policy']}) ===")
print(pd.DataFrame([dict(split=f"validation ({len(YDV)} cases)", **{k: round(m[k], 4) for k in ("macro_f1_present", "macro_f1_universe", "macro_f1_universe_zd1", "micro_f1", "subset_acc")})]).to_string(index=False))
print(f"selected macro-F1 [{MACRO_CONVENTION}]: {sel_f['scores'][sel_f['policy']]:.4f} (95% bootstrap interval of the fitted predictions {lo:.3f} - {hi:.3f}; a per-statute policy is scored by a 2-fold estimate, the others are fitted on validation) | TF-IDF baseline {globals().get('BASELINE', {}).get('macro_f1', float('nan')):.4f}")
print(f"statutes per case: validation true {card_dev:.2f}, validation predicted {Pd.sum(1).mean():.2f}, test predicted {Pt.sum(1).mean():.2f} | test file: {len(rids)} ids, 'pred' = integer statute ids")
if globals().get("BASELINE") and sel_f["scores"][sel_f["policy"]] < BASELINE["macro_f1"]: print("WARNING: the neural result is BELOW the TF-IDF baseline; the TF-IDF baseline's submission is a valid fallback:", SUBDIR / "BL_tfidf.json")
sup = YDV.sum(0); q = np.digitize(sup, np.quantile(sup[sup > 0], [0.25, 0.5, 0.75])) if (sup > 0).sum() > 4 else np.zeros(V, int); f1l = m["f1_per_label"]
print("validation F1 by statute frequency (quartiles of validation support; rare statutes drive macro-F1):", {f"Q{k+1}": (round(float(f1l[(q == k) & (sup > 0)].mean()), 3) if ((q == k) & (sup > 0)).any() else None) for k in range(4)})
print("the 5 worst frequent statutes:", [(LABELS[j], int(sup[j]), round(float(f1l[j]), 2)) for j in sorted(np.where(sup >= max(MIN_SUP, 1))[0], key=lambda j: f1l[j])[:5]])


In [ ]:
# ================= CELL 15 — THE SIX EXPERIMENTS THAT SATISFY THE CAPSTONE RULES + REQUIREMENTS REPORT  [RUNS after the experiments; trains nothing] =================
# Rules (as summarised in the capstone document): six substantially different experiments, all four Phase 1 architectures, at least five distinct adaptation techniques, compulsory LoRMA, at least one knowledge-distillation experiment.
# An experiment = a distinct (Phase 1 model, technique) pair. Re-runs, seeds, learning rates, losses and window settings are hyperparameters: they are ablations (control=True) and never counted; of runs sharing a pair, only the best is kept.
from itertools import combinations
FAMILY = {"full": "full fine-tuning", "partial": "partial fine-tuning", "lora": "LoRA", "rslora": "LoRA", "lorma_plus": "LoRMA", "lorma_pi": "LoRMA", "ia3": "IA3", "kd": "knowledge distillation"}
REQ_MODELS = ("gpt2_new", "modernbert", "t5", "bilstm")
assert LEDGER.exists(), "no experiment has finished yet"
led = pd.read_csv(LEDGER); led["control"] = led["control"].astype(str).str.lower().eq("true"); led["exp_id"] = led.get("exp_id", "").fillna("")
base_row = led[led.model == "tfidf"]; ablations = led[led.control & (led.model != "tfidf")]; cand = led[(~led.control) & (led.model != "tfidf") & led.technique.isin(FAMILY)].copy(); assert len(cand), "no counted experiment in the ledger yet"
cand["family"] = cand.technique.map(FAMILY); cand["pair"] = cand.model + "|" + cand.family; cand = cand.sort_values("dev_macro_f1", ascending=False).drop_duplicates("pair").reset_index(drop=True)          # one run per (model, technique)
def rule_check(sub):
    fam = set(sub.family); return dict(all_four_architectures=set(REQ_MODELS) <= set(sub.model), has_lorma="LoRMA" in fam, has_distillation="knowledge distillation" in fam, five_techniques=len(fam) >= 5, six_experiments=len(sub) >= 6, families=sorted(fam))
def choose_set(c, size=6):
    n = min(size, len(c)); best = None
    for combo in combinations(range(len(c)), n):
        sub = c.iloc[list(combo)]; rc = rule_check(sub); strict = all(rc[k] for k in ("all_four_architectures", "has_lorma", "has_distillation", "five_techniques", "six_experiments"))
        key = (strict, len(set(sub.model) & set(REQ_MODELS)), rc["has_lorma"], rc["has_distillation"], min(len(rc["families"]), 5), len(sub), float(sub.dev_macro_f1.mean()))
        if best is None or key > best[0]: best = (key, list(sub.name))
    return best
key_, FINAL = choose_set(cand); sub = cand.set_index("name").loc[FINAL].reset_index(); rc = rule_check(sub); strict = key_[0]
unmet = ([] if rc["six_experiments"] else [f"only {len(sub)} distinct experiment(s) so far (six needed)"]) + ([] if rc["all_four_architectures"] else [f"architectures not covered: {sorted(set(REQ_MODELS) - set(sub.model))}"]) + ([] if rc["has_lorma"] else ["no LoRMA experiment (E2)"]) + ([] if rc["has_distillation"] else ["no distillation experiment (E5)"]) + ([] if rc["five_techniques"] else [f"only {len(rc['families'])} technique families (five needed)"])
status = "ALL CAPSTONE RULES SATISFIED" if strict else "NOT YET COMPLIANT: " + "; ".join(unmet)
cols = ["name", "exp_id", "model", "technique", "trainable_M", "dev_macro_f1", "dev_macro_f1_present", "dev_macro_f1_universe", "dev_micro_f1", "dev_subset_acc"]
print(f"SELECTED EXPERIMENTS ({len(sub)}): {status}"); print(sub[cols].round(4).to_string(index=False)); print("technique families:", rc["families"], "| macro-F1 columns: 'dev_macro_f1' follows MACRO_CONVENTION =", MACRO_CONVENTION)
if not strict: print("-> run the missing experiments (E1-E6 are split over Part 1 and Part 2) and re-run this cell.")
FINAL_DIR = WORK / "final_six"; shutil.rmtree(FINAL_DIR, ignore_errors=True); FINAL_DIR.mkdir()
for k_, n_ in enumerate(FINAL, 1):
    validate_submission_ml(SUBDIR / f"{n_}.json", DATA["test"]["ids"]); (FINAL_DIR / f"submission_{k_}_{n_}.json").write_text((SUBDIR / f"{n_}.json").read_text())
def md_table(df):
    try: return df.to_markdown(index=False)
    except Exception: return "```\n" + df.to_string(index=False) + "\n```"
(FINAL_DIR / "requirements_report.md").write_text("# Phase 5 (statute identification): requirement check\n\n**" + status + "**\n\n" + md_table(sub[cols].round(4)) +
    f"\n\n- all four Phase 1 architectures: {rc['all_four_architectures']}\n- LoRMA: {rc['has_lorma']}\n- knowledge distillation: {rc['has_distillation']}\n- technique families ({len(rc['families'])}): {', '.join(rc['families'])}\n- the old 3,000-step GPT-2 is not used anywhere\n"
    f"\n## Evaluation conventions (the official evaluator's convention is unresolved; nothing here claims to be it)\nMacro-F1 [present_in_val] = mean F1 over statutes occurring in validation; [universe] = mean over all {V} statutes of train ∪ validation with F1 = 0 for a statute that never occurs and is never predicted; [universe_zd1] = the same with F1 = 1 there. Micro-F1 from summed TP/FP/FN. Thresholds were tuned on validation data only. Tuning/selection convention: {MACRO_CONVENTION}.\n"
    + (f"\n## Baseline (not a Phase 1 model)\n{md_table(base_row[['name', 'dev_macro_f1', 'dev_macro_f1_present', 'dev_macro_f1_universe', 'dev_micro_f1']].round(4))}\n" if len(base_row) else "")
    + (f"\n## Ablations (not counted as experiments)\n{md_table(ablations[['name', 'model', 'technique', 'dev_macro_f1', 'dev_micro_f1']].round(4))}\n" if len(ablations) else ""))
print(f"\nfiles: {FINAL_DIR}/ (the selected experiments' submissions + requirements_report.md). The final prediction.json (best single run or ensemble) is written by the salvage cell.")


In [ ]:
# ================= CELL 16 — BACKUP  [NEEDS Kaggle credentials for the upload, otherwise a dry run] =================
# Uploads by itself when the credentials cell verified the Kaggle key (KAGGLE_OK); otherwise it is a dry run that only lists what would be uploaded. It never asks you to type anything. A refresh or a new session loses /kaggle/working; this does not.
import hashlib, shutil, subprocess, getpass
DATASET_SLUG = "cs781-phase5-lsi-results"; INCLUDE_MODELS = True; DRY_RUN = not globals().get("KAGGLE_OK", False); BACKUP_OK = False          # real upload automatically when the credentials cell verified the Kaggle key; otherwise a dry run  # INCLUDE_MODELS=False skips the ~450 MB model files (then distillation teachers are not backed up)
if not DRY_RUN and not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
    try:
        from kaggle_secrets import UserSecretsClient
        _s = UserSecretsClient(); os.environ["KAGGLE_USERNAME"] = _s.get_secret("KAGGLE_USERNAME"); os.environ["KAGGLE_KEY"] = _s.get_secret("KAGGLE_KEY")
    except Exception:
        raise RuntimeError("Kaggle credentials not found: run the CREDENTIALS cell first, or add the Secrets KAGGLE_USERNAME and KAGGLE_KEY (Add-ons > Secrets, Attached ON). This cell never asks you to type them.")
OWNER = os.environ.get("KAGGLE_USERNAME", "oipranesh"); stage = Path("/kaggle/working/lsi_backup"); shutil.rmtree(stage, ignore_errors=True); stage.mkdir(parents=True); files = {}
for rd in sorted((WORK / "runs").glob("*")):                                   # flat names (the Kaggle CLI skips sub-folders): <run>__<file>
    if rd.name.startswith(("pilot", "probe", "smoke")): continue
    for f in sorted(rd.iterdir()):
        if f.name == "model.pt" and not INCLUDE_MODELS: continue
        files[f"{rd.name}__{f.name}"] = f
for f in sorted(CKPT.glob("*")):                                                 # the fine-tuned checkpoints (PEFT ones hold only the adapters + head)
    if f.suffix == ".pt" and not INCLUDE_MODELS: continue
    files[f"ckpt__{f.name}"] = f
for f in sorted(SUBDIR.glob("*.json")): files[f"submission__{f.name}"] = f
for f in (LEDGER, WORK / "order.json", Path("/kaggle/working/prediction.json"), Path("/kaggle/working/prediction_best_single.json"), Path("/kaggle/working/prediction_ensemble.json")):
    if f.exists(): files[f"top__{f.name}"] = f
assert files, "nothing to back up yet: run an experiment first"
for name, p in files.items():
    try: os.link(p, stage / name)
    except OSError: shutil.copy2(p, stage / name)
def sha256(p, chunk=1 << 24):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()
runs_in = sorted({n.split("__")[0] for n in files if not n.startswith(("submission__", "top__", "ckpt__"))}); manifest = {"runs": runs_in, "files": {n: {"bytes": (stage / n).stat().st_size, "sha256": sha256(stage / n)} for n in files}}
(stage / "MANIFEST.json").write_text(json.dumps(manifest, indent=1)); (stage / "dataset-metadata.json").write_text(json.dumps({"title": "CS781 Phase 5 LSI results backup", "id": f"{OWNER}/{DATASET_SLUG}", "licenses": [{"name": "unknown"}]}, indent=1))
tot = sum(p.stat().st_size for p in stage.iterdir()); print(f"staged {len(files) + 2} files, {tot/2**30:.2f} GiB; runs: {runs_in}")
for n in sorted(files): print(f"  {n:58s} {manifest['files'][n]['bytes']/2**20:9.1f} MiB")
kb = shutil.which("kaggle"); cmd_c = ["kaggle", "datasets", "create", "-p", str(stage)]; cmd_v = ["kaggle", "datasets", "version", "-p", str(stage), "-m", f"runs: {', '.join(runs_in)}"]
if DRY_RUN or not kb: print("\nDRY RUN: would run", " ".join(cmd_c), "(or, if the dataset exists)", " ".join(cmd_v), "\nSet DRY_RUN = False and run again to upload.")
else:
    chk = subprocess.run(["kaggle", "datasets", "list", "--mine"], capture_output=True, text=True, timeout=180)
    if chk.returncode != 0: print("credentials check FAILED (Internet on? legacy key?):", (chk.stdout + chk.stderr)[:300], "\nNothing was uploaded.")
    else:
        r = subprocess.run(cmd_c, capture_output=True, text=True); o = (r.stdout + r.stderr).strip(); print("\n".join(l for l in o.splitlines() if "%|" not in l)[-1200:])
        if r.returncode != 0 and "already" in o.lower(): print("dataset exists -> new version"); r = subprocess.run(cmd_v, capture_output=True, text=True); print("\n".join(l for l in (r.stdout + r.stderr).splitlines() if "%|" not in l)[-1200:])
        BACKUP_OK = r.returncode == 0; print("\nBACKUP OK" if r.returncode == 0 else f"\nBACKUP FAILED (exit {r.returncode}); the staged folder {stage} is still there")


In [ ]:
# ================= CELL 17 — WHAT 'Save & Run All' KEEPS AS THIS NOTEBOOK'S OUTPUT, a format check, and a size cleanup  [RUNS NOW] =================
DELETE_BULKY = True       # remove the tokenised cache (and the staged backup copy after a successful upload) from the output; predictions, probabilities, results and checkpoints are kept
def size_gb(p): return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file()) / 2**30 if Path(p).exists() else 0.0
print(f"/kaggle/working before cleanup: {size_gb('/kaggle/working'):.2f} GiB (cache {size_gb(WORK / 'cache'):.2f}, checkpoints {size_gb(CKPT):.2f}, runs {size_gb(WORK / 'runs'):.2f}, submissions {size_gb(SUBDIR):.3f})")
if DELETE_BULKY: shutil.rmtree(WORK / "cache", ignore_errors=True)
if globals().get("BACKUP_OK"): shutil.rmtree("/kaggle/working/lsi_backup", ignore_errors=True); print("backup uploaded: the staged copy was removed from the output")
print(f"after cleanup: {size_gb('/kaggle/working'):.2f} GiB\n\nPREDICTION FILES (the capstone format: a JSON list of {{\"id\": ..., \"pred\": [integer statute ids]}}):")
for f in sorted(SUBDIR.glob("*.json")) + [Path("/kaggle/working") / n for n in ("prediction.json", "prediction_best_single.json")]:
    if f.exists(): sub_ = json.load(open(f)); print(f"  {str(f.relative_to('/kaggle/working')):42s} {len(sub_)} cases | {np.mean([len(s[PRED_KEY]) for s in sub_]):.2f} statutes per case")
pj = Path("/kaggle/working/prediction.json")
if pj.exists(): print("\nfirst two objects of prediction.json:", json.load(open(pj))[:2]); validate_submission_ml(pj, DATA["test"]["ids"]); print("format validated against the test ids and the label universe")
else: print("\nWARNING: /kaggle/working/prediction.json does not exist: run the salvage cell (it needs at least one finished experiment). The baseline's fallback submission is", SUBDIR / "BL_tfidf.json")
print(f"checkpoints ({CKPT}):", sorted(p.name for p in CKPT.glob('*.pt')))
